# 03 — Analyze (Fase 4): Insights (SQL con DuckDB)

**Input:** `../data/clean/hotel_bookings_clean.csv` (86.999 filas) + preguntas guía de la Fase 1.
**Output:** respuesta a cada pregunta guía con soporte de datos.

**Herramienta:** SQL dentro del notebook con **DuckDB directo** — `import duckdb` y luego `duckdb.sql("SELECT ...")`. Sin magics ni conexiones. DuckDB consulta el CSV directamente (`FROM '../data/clean/...csv'`) o un DataFrame de pandas por su nombre de variable.

> `reservation_status` y `reservation_status_date` quedan **excluidas** de todo el análisis (data leakage). Foco: **City Hotel**.

## 0. Setup

Cargar el CSV y dejar vistas de trabajo: `bookings` (todo) y `city` (solo City Hotel).

In [159]:
import duckdb as dd
from IPython.display import display
# Crea la vista 'bookings': duckdb.sql("CREATE OR REPLACE VIEW bookings AS SELECT * FROM '../data/clean/hotel_bookings_clean.csv'")
dd.sql("CREATE OR REPLACE VIEW bookings AS SELECT * FROM '../data/clean/hotel_bookings_clean.csv'")
# Crea la vista 'city' filtrando hotel = 'City Hotel'
dd.sql("CREATE OR REPLACE VIEW city AS SELECT * FROM bookings WHERE hotel = 'City Hotel'")
# Verifica con un SELECT COUNT(*): ¿cuántas filas tiene bookings y cuántas city?
display(dd.sql("SELECT (SELECT COUNT(*) FROM bookings) AS total_bookings, (SELECT COUNT(*) FROM city) AS total_city").df().style.hide(axis='index'))

total_bookings,total_city
86999,53050


## 1. Estadísticas descriptivas

Media, mediana, mín, máx y desviación estándar de `lead_time`, `adr` y `total_of_special_requests`, más la tasa global de cancelación del City Hotel.

In [160]:
# SELECT con AVG, MEDIAN, MIN, MAX, STDDEV para cada variable clave de la vista 'city' (valores de referencia)
display(
    dd.sql("""
        SELECT
            'lead_time' AS variable,
            ROUND(AVG(lead_time), 2)    AS media,
            ROUND(MEDIAN(lead_time), 2) AS mediana,
            MIN(lead_time)              AS minimo,
            MAX(lead_time)              AS maximo,
            ROUND(STDDEV(lead_time), 2) AS desv_std
        FROM city

        UNION ALL

        SELECT
            'adr',
            ROUND(AVG(adr), 2),
            ROUND(MEDIAN(adr), 2),
            MIN(adr),
            MAX(adr),
            ROUND(STDDEV(adr), 2)
        FROM city

        UNION ALL

        SELECT
            'special_requests',
            ROUND(AVG(total_of_special_requests), 2),
            ROUND(MEDIAN(total_of_special_requests), 2),
            MIN(total_of_special_requests),
            MAX(total_of_special_requests),
            ROUND(STDDEV(total_of_special_requests), 2)
        FROM city
    """).df()
    .style
    .format({
        "media":    "{:.2f}",
        "mediana":  "{:.2f}",
        "minimo":   "{:.0f}",
        "maximo":   "{:.0f}",
        "desv_std": "{:.2f}",
    })
    .hide(axis="index")
)

display(
    dd.sql("""
        SELECT ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct
        FROM city
    """).df()
    .style
    .format({"cancel_rate_pct": "{:.2f} %"})
    .hide(axis="index")
)

variable,media,mediana,minimo,maximo,desv_std
lead_time,77.61,50.00,0,629,82.01
adr,111.30,105.40,0,510,41.93
special_requests,0.71,1.00,0,5,0.83


cancel_rate_pct
30.07 %


## 2. Pregunta 1 — ¿Qué tasa de cancelación hay y cómo varía por mes?

**Métrica:** % de `is_canceled` por `arrival_date_month`.
**Decisión:** dónde reforzar overbooking/políticas.

In [161]:
# Tasa de cancelación por mes, desglosada por año y ordenada cronológicamente
display(
    dd.sql("""
        SELECT
            arrival_date_month,
            ROUND(AVG(CASE WHEN arrival_date_year = 2015 THEN is_canceled END) * 100, 1) AS pct_2015,
            ROUND(AVG(CASE WHEN arrival_date_year = 2016 THEN is_canceled END) * 100, 1) AS pct_2016,
            ROUND(AVG(CASE WHEN arrival_date_year = 2017 THEN is_canceled END) * 100, 1) AS pct_2017,
            ROUND(AVG(is_canceled) * 100, 1)  AS pct_total,
            SUM(is_canceled)                  AS canceladas_total,
            COUNT(*)                          AS reservas_total
        FROM city
        GROUP BY arrival_date_month
        ORDER BY
            CASE arrival_date_month
                WHEN 'January' THEN 1  WHEN 'February' THEN 2  WHEN 'March' THEN 3
                WHEN 'April' THEN 4    WHEN 'May' THEN 5       WHEN 'June' THEN 6
                WHEN 'July' THEN 7     WHEN 'August' THEN 8    WHEN 'September' THEN 9
                WHEN 'October' THEN 10 WHEN 'November' THEN 11 WHEN 'December' THEN 12
            END
    """).df()
    .style
    .format({
        "pct_2015": "{:.1f} %", "pct_2016": "{:.1f} %",
        "pct_2017": "{:.1f} %", "pct_total": "{:.1f} %",
        "canceladas_total": "{:.0f}",
}, na_rep="—")
    .hide(axis="index")
)

arrival_date_month,pct_2015,pct_2016,pct_2017,pct_total,canceladas_total,reservas_total
January,—,20.8 %,32.4 %,28.1 %,759,2705
February,—,22.7 %,30.8 %,27.1 %,967,3573
March,—,27.3 %,28.9 %,28.1 %,1353,4816
April,—,31.1 %,37.5 %,34.4 %,1739,5051
May,—,27.5 %,36.0 %,32.3 %,1739,5386
June,—,26.8 %,33.5 %,30.4 %,1517,4986
July,59.0 %,27.3 %,34.6 %,33.1 %,1887,5698
August,20.9 %,32.4 %,36.3 %,32.1 %,2098,6544
September,18.8 %,29.2 %,—,25.1 %,1059,4220
October,18.6 %,31.8 %,—,26.9 %,1125,4180


## 3. Pregunta 2 — ¿La antelación (lead time) predice cancelación?

**Métrica:** cancel rate por tramo de `lead_time`.
**Decisión:** cuándo pedir garantía/depósito.

Tramos: 0–7, 8–30, 31–90, 91–180, 180+ días.

In [162]:
# Cancel rate por tramo de lead_time (0-7, 8-30, 31-90, 91-180, 180+)
display(
    dd.sql("""
        SELECT
            CASE
                WHEN lead_time <= 7   THEN '0-7'
                WHEN lead_time <= 30  THEN '8-30'
                WHEN lead_time <= 90  THEN '31-90'
                WHEN lead_time <= 180 THEN '91-180'
                ELSE '180+'
            END AS tramo_lead_time,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        GROUP BY tramo_lead_time
        ORDER BY MIN(lead_time)
    """).df()
    .style
    .format({
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)

tramo_lead_time,cancel_rate_pct,canceladas,reservas
0-7,10.51 %,"1,028","9,782"
8-30,27.86 %,"2,922","10,488"
31-90,33.17 %,"5,008","15,100"
91-180,36.72 %,"4,223","11,502"
180+,44.87 %,"2,772","6,178"


## 4. Pregunta 3 — ¿Qué segmento/canal cancela más?

**Métrica:** cancel rate por `market_segment` y por `distribution_channel`.
**Decisión:** dónde ajustar condiciones por canal.

Los NaN mantenidos a propósito en la Fase 3 (cambio #14) se excluyen al agrupar con `WHERE market_segment IS NOT NULL`.

In [163]:
# Cancel rate por market_segment
display(
    dd.sql("""
        SELECT
            market_segment,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        WHERE market_segment IS NOT NULL
        GROUP BY market_segment
        ORDER BY cancel_rate_pct DESC
    """).df()
    .style
    .format({
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)

# Cuidado con los grupos pequeños: una tasa alta con n=20 no es comparable a una con n=20.000 (por eso se deja el COUNT visible)

market_segment,cancel_rate_pct,canceladas,reservas
Online TA,36.10 %,"12,543","34,743"
Groups,33.87 %,887,"2,619"
Aviation,19.91 %,45,226
Offline TA/TO,17.35 %,"1,255","7,235"
Direct,16.43 %,905,"5,509"
Corporate,11.86 %,263,"2,217"
Complementary,10.62 %,53,499


In [164]:
# Cancel rate por distribution_channel
display(
    dd.sql("""
        SELECT
            distribution_channel,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        WHERE distribution_channel IS NOT NULL
        GROUP BY distribution_channel
        ORDER BY cancel_rate_pct DESC
    """).df()
    .style
    .format({
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)

distribution_channel,cancel_rate_pct,canceladas,reservas
TA/TO,33.04 %,"14,620","44,252"
GDS,19.89 %,36,181
Direct,15.99 %,963,"6,023"
Corporate,12.74 %,330,"2,590"


In [165]:
# Peso de cada segmento sobre el total de cancelaciones (¿quién aporta más canceladas?)
display(
    dd.sql("""
        SELECT
            market_segment,
            SUM(is_canceled) AS canceladas,
            ROUND(SUM(is_canceled) * 100.0 / (SELECT SUM(is_canceled) FROM city), 2) AS pct_del_total_cancelaciones
        FROM city
        WHERE market_segment IS NOT NULL
        GROUP BY market_segment
        ORDER BY canceladas DESC
    """).df()
    .style
    .format({"pct_del_total_cancelaciones": "{:.2f} %", "canceladas": "{:,.0f}"})
    .hide(axis="index")
)

market_segment,canceladas,pct_del_total_cancelaciones
Online TA,"12,543",78.62 %
Offline TA/TO,"1,255",7.87 %
Direct,905,5.67 %
Groups,887,5.56 %
Corporate,263,1.65 %
Complementary,53,0.33 %
Aviation,45,0.28 %


In [166]:
# Online TA vs todo el resto: ¿cuánto más cancela?
display(
    dd.sql("""
        SELECT
            CASE WHEN market_segment = 'Online TA' THEN 'Online TA' ELSE 'Resto' END AS grupo,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        WHERE market_segment IS NOT NULL
        GROUP BY grupo
    """).df()
    .style
    .format({"cancel_rate_pct": "{:.2f} %", "canceladas": "{:,.0f}", "reservas": "{:,.0f}"})
    .hide(axis="index")
)

grupo,cancel_rate_pct,canceladas,reservas
Online TA,36.10 %,"12,543","34,743"
Resto,18.62 %,"3,408","18,305"


## 5. Pregunta 4 — ¿El depósito reduce cancelaciones?

**Métrica:** cancel rate por `deposit_type`.
**Decisión:** si exigir depósito.

El resultado es contraintuitivo: las reservas Non Refund cancelan más. Antes de concluir, se investiga qué tipo de reservas usan 'Non Refund' (cruce con segmento y lead time en las celdas siguientes).

In [167]:
# Cancel rate por deposit_type
display(
    dd.sql("""
        SELECT
            deposit_type,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        GROUP BY deposit_type
        ORDER BY cancel_rate_pct DESC
    """).df()
    .style
    .format({
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)

# Cruce deposit_type × market_segment: ¿quién usa cada tipo de depósito?
display(
    dd.sql("""
        SELECT
            deposit_type,
            market_segment,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        WHERE market_segment IS NOT NULL
        GROUP BY deposit_type, market_segment
        ORDER BY deposit_type, reservas DESC
    """).df()
    .style
    .format({
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)

deposit_type,cancel_rate_pct,canceladas,reservas
Non Refund,97.16 %,820,844
Refundable,66.67 %,10,15
No Deposit,28.98 %,"15,123","52,191"


deposit_type,market_segment,cancel_rate_pct,canceladas,reservas
No Deposit,Online TA,36.07 %,"12,527","34,725"
No Deposit,Offline TA/TO,14.17 %,987,"6,964"
No Deposit,Direct,16.31 %,897,"5,499"
No Deposit,Corporate,10.37 %,224,"2,161"
No Deposit,Groups,18.35 %,388,"2,115"
No Deposit,Complementary,10.62 %,53,499
No Deposit,Aviation,19.91 %,45,226
Non Refund,Groups,99.20 %,499,503
Non Refund,Offline TA/TO,99.63 %,268,269
Non Refund,Corporate,71.70 %,38,53


In [168]:
# Antelación por deposit_type: ¿las Non Refund son reservas de mucho lead time?
display(
    dd.sql("""
        SELECT
            deposit_type,
            ROUND(AVG(lead_time), 1)    AS lead_time_medio,
            ROUND(MEDIAN(lead_time), 1) AS lead_time_mediana,
            COUNT(*)                    AS reservas
        FROM city
        GROUP BY deposit_type
        ORDER BY lead_time_medio DESC
    """).df()
    .style
    .format({
        "lead_time_medio":   "{:.1f}",
        "lead_time_mediana": "{:.1f}",
        "reservas":          "{:,.0f}",
    })
    .hide(axis="index")
)

deposit_type,lead_time_medio,lead_time_mediana,reservas
Non Refund,223.7,198.5,844
Refundable,82.5,52.0,15
No Deposit,75.2,49.0,"52,191"


## 6. Pregunta 5 — ¿ADR o peticiones especiales se relacionan con cancelar?

**Métrica:** cancel rate por tramo de `adr` y por nº de `total_of_special_requests`.
**Decisión:** pricing / detección de riesgo.

In [169]:
# Cancel rate por cuartil de ADR. Usamos NTILE(4) para crear cuartiles de ADR una vez ordenados de forma ascendente.
display(
    dd.sql("""
        WITH q AS (
            SELECT
                is_canceled,
                NTILE(4) OVER (ORDER BY adr) AS adr_quartile,
                adr
            FROM city
        )
        SELECT
            adr_quartile,
            MIN(adr)                         AS adr_min,
            MAX(adr)                         AS adr_max,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM q
        GROUP BY adr_quartile
        ORDER BY adr_quartile
    """).df()
    .style
    .format({
        "adr_min":         "{:.2f}",
        "adr_max":         "{:.2f}",
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)

adr_quartile,adr_min,adr_max,cancel_rate_pct,canceladas,reservas
1,0.00,84.33,23.33 %,"3,094","13,263"
2,84.33,105.40,29.03 %,"3,850","13,263"
3,105.40,134.10,33.17 %,"4,399","13,262"
4,134.10,510.00,34.76 %,"4,610","13,262"


In [170]:
# Cancel rate por nº de special_requests
display(
    dd.sql("""
        SELECT
            total_of_special_requests AS special_requests,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        GROUP BY total_of_special_requests
        ORDER BY total_of_special_requests
    """).df()
    .style
    .format({
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)


special_requests,cancel_rate_pct,canceladas,reservas
0,38.36 %,"10,049","26,197"
1,23.00 %,"4,121","17,914"
2,20.82 %,"1,516","7,281"
3,16.91 %,246,"1,455"
4,11.05 %,20,181
5,4.55 %,1,22


In [171]:
# 0 peticiones vs 1+: tasa, peso en volumen y peso sobre el total de cancelaciones
display(
    dd.sql("""
        SELECT
            CASE WHEN total_of_special_requests = 0 THEN '0' ELSE '1+' END AS peticiones,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas,
            ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM city), 2)               AS pct_del_volumen,
            ROUND(SUM(is_canceled) * 100.0 / (SELECT SUM(is_canceled) FROM city), 2) AS pct_de_cancelaciones
        FROM city
        GROUP BY peticiones
        ORDER BY peticiones
    """).df()
    .style
    .format({
        "cancel_rate_pct":      "{:.2f} %",
        "canceladas":           "{:,.0f}",
        "reservas":             "{:,.0f}",
        "pct_del_volumen":      "{:.2f} %",
        "pct_de_cancelaciones": "{:.2f} %",
    })
    .hide(axis="index")
)

peticiones,cancel_rate_pct,canceladas,reservas,pct_del_volumen,pct_de_cancelaciones
0,38.36 %,"10,049","26,197",49.38 %,62.99 %
1+,21.99 %,"5,904","26,853",50.62 %,37.01 %


## 7. Señales de cliente — repeat guest y cancelaciones previas

**Alcance Fase 1:** "señales de cliente (depósito, peticiones especiales, **repeat guest**)".
**Métrica:** cancel rate por `is_repeated_guest` y por `previous_cancellations`.

In [172]:
# Cancel rate por is_repeated_guest
display(
    dd.sql("""
        SELECT
            is_repeated_guest,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        GROUP BY is_repeated_guest
        ORDER BY is_repeated_guest
    """).df()
    .style
    .format({
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)

# Cancel rate por previous_cancellations (agrupado 0, 1, 2+ por la cola larga)
display(
    dd.sql("""
        SELECT
            CASE
                WHEN previous_cancellations = 0 THEN '0'
                WHEN previous_cancellations = 1 THEN '1'
                ELSE '2+'
            END AS cancelaciones_previas,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        GROUP BY cancelaciones_previas
        ORDER BY cancelaciones_previas
    """).df()
    .style
    .format({
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)
# ¿El huésped repetidor cancela menos? ¿El que ya canceló antes reincide?

is_repeated_guest,cancel_rate_pct,canceladas,reservas
0,30.67 %,"15,765","51,394"
1,11.35 %,188,"1,656"


cancelaciones_previas,cancel_rate_pct,canceladas,reservas
0,29.20 %,"15,148","51,878"
1,80.35 %,773,962
2+,15.24 %,32,210


In [173]:
# Cruce previous_cancellations × deposit_type: ¿el grupo "1 previa" es otra cara de Non Refund?
display(
    dd.sql("""
        SELECT
            CASE
                WHEN previous_cancellations = 0 THEN '0'
                WHEN previous_cancellations = 1 THEN '1'
                ELSE '2+'
            END AS cancelaciones_previas,
            deposit_type,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        GROUP BY cancelaciones_previas, deposit_type
        ORDER BY cancelaciones_previas, reservas DESC
    """).df()
    .style
    .format({
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)

cancelaciones_previas,deposit_type,cancel_rate_pct,canceladas,reservas
0,No Deposit,28.33 %,"14,504","51,205"
0,Non Refund,96.35 %,634,658
0,Refundable,66.67 %,10,15
1,No Deposit,75.68 %,588,777
1,Non Refund,100.00 %,185,185
2+,No Deposit,14.83 %,31,209
2+,Non Refund,100.00 %,1,1


## 8. Tendencias, patrones y anomalías

Consolidar: ¿qué 3–5 hallazgos tienen soporte más fuerte? ¿Algo sorprendente o contradictorio? (candidatos: deposit_type, meses pico, lead time).

In [174]:
# Cruce de máximo riesgo: Online TA × lead time >90 (umbral propuesto en P2)
display(
    dd.sql("""
        SELECT
            CASE WHEN market_segment = 'Online TA' THEN 'Online TA' ELSE 'Resto' END
                || ' · ' ||
            CASE WHEN lead_time > 90 THEN 'lead >90' ELSE 'lead 0-90' END AS grupo,
            ROUND(AVG(is_canceled) * 100, 2) AS cancel_rate_pct,
            SUM(is_canceled)                 AS canceladas,
            COUNT(*)                         AS reservas
        FROM city
        WHERE market_segment IS NOT NULL
        GROUP BY grupo
        ORDER BY cancel_rate_pct DESC
    """).df()
    .style
    .format({
        "cancel_rate_pct": "{:.2f} %",
        "canceladas":      "{:,.0f}",
        "reservas":        "{:,.0f}",
    })
    .hide(axis="index")
)

grupo,cancel_rate_pct,canceladas,reservas
Online TA · lead >90,44.70 %,"5,419","12,122"
Online TA · lead 0-90,31.49 %,"7,124","22,621"
Resto · lead >90,28.36 %,"1,576","5,558"
Resto · lead 0-90,14.37 %,"1,832","12,747"


### Los 5 hallazgos con soporte más fuerte (ordenados por fuerza × accionabilidad)

1. **Perfil de máximo riesgo — Online TA + lead time >90 días:** 44,70% de cancelación (3,1× el suelo de 14,37%), 23% del volumen y 34% de todas las cancelaciones. Las señales se potencian, no se suman: el lead time largo solo es peligroso en Online TA. *(Candidato a titular de Fase 5; incluye dentro el hallazgo de P3: Online TA = 79% de las cancelaciones.)*
2. **Cancelación previa:** quien ya canceló una vez cancela al 80,35% (75,68% incluso sin depósito → 2,7× la base, señal independiente de P4). La más accionable por reserva: el hotel la conoce en el momento de reservar.
3. **Lead time:** crecimiento monotónico 10,51% (0–7 días) → 44,87% (180+). El predictor continuo más claro; umbral propuesto para pedir garantía: 90+ días.
4. **Cero peticiones especiales:** 38,36% vs 21,99% con 1+; concentra el 63% de las cancelaciones con el 49% del volumen. Gran cobertura como señal de scoring (no es palanca causal).
5. **Non Refund 97,16% (anomalía, lección causal):** no es efecto del depósito sino selección — 91% son Groups/Offline TA/TO con mediana de 198,5 días de antelación. Vale como advertencia metodológica: asociación ≠ causalidad.

**Quedan fuera del top:** estacionalidad (débil, solo abril estable), ADR (señal débil y probablemente no independiente), repeat guest (fiable pero alcance del 3% del volumen) y la anomalía del 2+ (n=210, solo curiosidad).


## 9. Resumen de hallazgos

Base de referencia: **City Hotel, 53.050 reservas, tasa de cancelación 30,07%** (jul 2015 – ago 2017).

| # | Pregunta | Hallazgo (1 línea) | Evidencia | Importancia |
|---|---|---|---|---|
| 1 | Tasa por mes | Estacionalidad parcial/débil: solo abril supera la base en todos los años con datos (31,1% en 2016, 37,5% en 2017); el volumen no explica la tasa | Tabla mes × año (P1) | Media |
| 2 | Lead time | La tasa crece monotónicamente con la antelación: 10,51% (0–7 días) → 44,87% (180+), más de 4×; umbral accionable propuesto: 90+ días | Tabla tramos lead time (P2) | Alta |
| 3 | Segmento/canal | Online TA cancela al 36,10% vs 18,62% el resto (1,9×) y concentra el 79% de todas las cancelaciones con el 65% del volumen; el canal no aporta señal independiente | Tablas segment + Online TA vs resto (P3) | Alta |
| 4 | Depósito | Non Refund cancela al 97,16% — pero es selección, no efecto del depósito: 844 reservas (1,6%), 91% de Groups y Offline TA/TO, con lead time mediano de 198,5 días vs 49 de No Deposit | Tablas deposit_type, cruce × segment, lead time × deposit (P4) | Media (anomalía clave) |
| 5 | ADR / peticiones | ADR: señal débil (23,33% Q1 → 34,76% Q4). Peticiones: señal fuerte — 0 peticiones cancela al 38,36% vs 21,99% con 1+, y concentra el 63% de las cancelaciones con el 49% del volumen | Tablas cuartiles ADR, special_requests, 0 vs 1+ (P5) | Alta (peticiones) |
| 6 | Repeat guest / canc. previas | Cancelación previa = señal más accionable: 80,35% (75,68% incluso sin depósito, 2,7× la base); repeat guest cancela a poco más de un tercio de la base (11,35%, n=1.656) | Tablas is_repeated_guest, previous_cancellations 0/1/2+, cruce × deposit (P6) | Alta |
| 7 | Cruce P2×P3 | Perfil de máximo riesgo: Online TA + lead >90 cancela al 44,70% (3,1× el suelo de 14,37%) — 23% del volumen y 34% de todas las cancelaciones; las señales se potencian, no se suman | Tabla cruce Online TA × lead time (bloque 8) | Alta |

**Anomalías detectadas:** Non Refund 97,16% (selección de reservas de riesgo, no efecto del depósito — lección causal del análisis) · 2+ cancelaciones previas cancela al 15,24% (n=210, rompe la monotonía; hipótesis no verificable: clientes de alta frecuencia).

**Limitaciones:**
- **Asociación ≠ causalidad** en todo el análisis: datos observacionales. Ejemplos concretos: el depósito no causa cancelación (P4, sesgo de selección) y las peticiones no causan retención (P5, señalan compromiso).
- **Cobertura temporal:** jul 2015 – ago 2017; 2015 y 2017 son años incompletos (afecta a las comparaciones mensuales de P1).
- **Alcance:** un solo hotel urbano (City Hotel, Lisboa); los hallazgos no son generalizables a otros hoteles ni al Resort Hotel sin re-análisis.
- **ADR como señal no independiente:** puede reflejar temporada y mezcla de segmento (hipótesis no contrastada).
- **Leakage controlado:** `reservation_status` y `reservation_status_date` excluidas de todo el análisis; NaN de Fase 3 excluidos vía `IS NOT NULL` (2 filas en `market_segment`, 4 en `distribution_channel`).

Las queries finales, en SQL puro, están en `sql/04_analyze.sql`.
